# 03 — Phase 3: Luật kết hợp

Phase 3 — Luật kết hợp thời tiết → công suất (`scripts/association.py`).

Giờ ban ngày, rời rạc hóa Low/Medium/High theo tertile từng hệ. Chạy cả Apriori và FP-Growth (min_support 0,05), 2 thuật toán cho cùng frequent itemsets; giữ luật có lift > 1 và confidence ≥ 0,5.

In [1]:
%matplotlib inline
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# Thư mục gốc của repo (notebook nằm trong notebooks/)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print("Repo root:", ROOT)
print("Python:", sys.executable)

Repo root: /home/hatch/workspace/data-mining-project/repo
Python: /home/hatch/workspace/data-mining-project/.venv/bin/python


In [2]:
# Chạy script tương ứng (dùng đúng Python của kernel này)
import subprocess
r = subprocess.run([sys.executable, str(ROOT / "scripts" / "association.py")],
                   capture_output=True, text=True)
print(r.stdout[-3000:])
print(r.stderr[-1000:] if r.returncode else "OK, exit 0")
assert r.returncode == 0, "script chạy lỗi, xem stderr ở trên"

System 4: 3736 giờ ban ngày dùng được, 39 frequent itemsets, 26 luật (lift>1, conf>=0.5), luật power mạnh nhất lift=2.99
System 1278: 3778 giờ ban ngày dùng được, 159 frequent itemsets, 111 luật (lift>1, conf>=0.5), luật power mạnh nhất lift=2.9
System 1403: 3818 giờ ban ngày dùng được, 31 frequent itemsets, 42 luật (lift>1, conf>=0.5), luật power mạnh nhất lift=3.0
Report: outputs/association_report.json, outputs/association_rules.csv

OK, exit 0


## 1. Có bao nhiêu luật?

In [3]:
import json
rep = json.load(open(ROOT / "outputs/association_report.json"))
for r in rep:
    print(f"System {r['system_id']}: {r['n_rules_lift_gt_1_conf_ge_0_5']} luật, "
          f"frequent itemsets Apriori {r['n_frequent_itemsets_apriori']} = "
          f"FP-Growth {r['n_frequent_itemsets_fpgrowth']}, "
          f"giờ ban ngày dùng được: {r['transactions_used']}")

System 4: 26 luật, frequent itemsets Apriori 39 = FP-Growth 39, giờ ban ngày dùng được: 3736
System 1278: 111 luật, frequent itemsets Apriori 159 = FP-Growth 159, giờ ban ngày dùng được: 3778
System 1403: 42 luật, frequent itemsets Apriori 31 = FP-Growth 31, giờ ban ngày dùng được: 3818


## 2. 10 luật mạnh nhất mỗi hệ (theo lift)

In [4]:
rules = pd.read_csv(ROOT / "outputs/association_rules.csv")
for sid in [4, 1278, 1403]:
    print(f"--- System {sid} ---")
    top = (rules[rules["system_id"] == sid]
           .sort_values("lift", ascending=False).head(10))
    print(top[["antecedents", "consequents", "support",
               "confidence", "lift"]].round(3).to_string(index=False))

--- System 4 ---
            antecedents  consequents  support  confidence  lift
 power=Low, temp=Medium      irr=Low    0.094       1.000 2.998
  irr=High, temp=Medium   power=High    0.116       0.998 2.994
  power=High, temp=High     irr=High    0.135       0.990 2.971
  irr=Medium, temp=High power=Medium    0.124       0.987 2.962
    irr=High, temp=High   power=High    0.135       0.982 2.948
   power=Low, temp=High      irr=Low    0.064       0.980 2.937
             power=High     irr=High    0.318       0.954 2.863
               irr=High   power=High    0.318       0.954 2.863
power=High, temp=Medium     irr=High    0.116       0.950 2.850
irr=Medium, temp=Medium power=Medium    0.106       0.945 2.836
--- System 1278 ---
             antecedents consequents  support  confidence  lift
 humidity=Low, power=Low     irr=Low    0.080       0.974 2.922
    power=Low, temp=High     irr=Low    0.073       0.968 2.904
       irr=Low, wind=Low   power=Low    0.125       0.967 2.898
  p

**Luật đáng nhớ:** `irr=High → power=High` đúng ở cả 3 hệ (confidence 0,92–0,95, lift 2,76–2,86). Ở Golden có dấu hiệu nhiệt làm giảm hiệu suất: bức xạ trung bình + nhiệt cao → công suất chỉ trung bình (confidence 0,987).